# RadJEPA-3D `.npy` volume viewer

Visualizza alcuni volumi `.npy` prodotti da `scripts/prepare_radjepa_volumes.py` (Step D della pipeline NLST), per un controllo visivo rapido (orientamento, finestratura HU, shape). Non carica il modello RadJEPA-3D: e' solo un viewer, cosi' funziona anche mentre Step D e' ancora in esecuzione sul server.

Formato atteso per ogni file: `[C, D, H, W] = [1, 32, 256, 256]`, float32, normalizzato in `[0, 1]`, orientamento canonico RAS.

In [ ]:
from pathlib import Path
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent

# Modifica questi path se vuoi puntare a un altro cohort/tag.
DATASET_ROOT = Path("/ssd2/domenico/datasets/nlst")
TAG = "pilot_1460_balanced"

npy_root = DATASET_ROOT / "processed/nlst" / TAG / "radjepa_npy"
npy_manifest_path = DATASET_ROOT / "processed/nlst" / TAG / "radjepa_npy_manifest.csv"

print("repo_root:", repo_root)
print("npy_root:", npy_root, npy_root.exists())
print("npy_manifest_path:", npy_manifest_path, npy_manifest_path.exists())

## 1. Elenca i volumi disponibili

`prepare_radjepa_volumes.py` scrive il manifest CSV solo alla fine dell'intero run, quindi mentre Step D e' ancora in corso il manifest potrebbe non esistere ancora. In quel caso si fa fallback a un glob diretto dei `.npy` gia' scritti su disco.

In [ ]:
if npy_manifest_path.exists():
    manifest = pd.read_csv(npy_manifest_path, dtype=str)
    if "radjepa_status" in manifest.columns:
        manifest = manifest[manifest["radjepa_status"].isin(["prepared", "exists"])]
    candidates = [
        (row["patient_id"], row["timepoint"], row["series_uid"], Path(row["radjepa_npy_path"]))
        for _, row in manifest.iterrows()
        if str(row.get("radjepa_npy_path", "")).strip()
    ]
    print(f"Loaded {len(candidates)} rows from manifest.")
else:
    paths = sorted(npy_root.glob("*/*.npy"))
    candidates = []
    for path in paths:
        stem = path.stem  # {patient_id}_{timepoint}_{series_uid}
        patient_id, timepoint, series_uid = stem.split("_", 2)
        candidates.append((patient_id, timepoint, series_uid, path))
    print(f"Manifest not found yet (Step D still running?) - found {len(candidates)} .npy files on disk via glob.")

assert candidates, "No .npy volumes found yet - wait for prepare_radjepa_volumes.py to write at least one file."

## 2. Campiona qualche volume e mostra le slice assiali

In [ ]:
N_SAMPLES = 6
SEED = 0

rng = random.Random(SEED)
sample = rng.sample(candidates, k=min(N_SAMPLES, len(candidates)))

for patient_id, timepoint, series_uid, path in sample:
    volume = np.load(path).astype(np.float32, copy=False)
    print(
        f"patient={patient_id} timepoint={timepoint} shape={volume.shape} "
        f"dtype={volume.dtype} min/mean/max={volume.min():.3f}/{volume.mean():.3f}/{volume.max():.3f}"
    )
    ct = volume[0]
    slice_ids = np.linspace(0, ct.shape[0] - 1, 8).astype(int)

    fig, axes = plt.subplots(2, 4, figsize=(12, 6))
    for ax, z in zip(axes.ravel(), slice_ids):
        # After the prepare_radjepa_volumes.py fix, ct[z] is ALREADY a
        # correctly oriented axial slice (anterior up, patient's right on
        # screen-left) with the plain array -- verified visually against
        # real anatomy (vertebral body posterior/bottom, heart asymmetry
        # left/screen-right). No origin change or flip needed here.
        ax.imshow(ct[z], cmap="gray", vmin=0.0, vmax=1.0)
        ax.set_title(f"z={z}")
        ax.axis("off")
    fig.suptitle(f"{patient_id} / {timepoint} - axial slices")
    plt.tight_layout()
    plt.show()

## 3. Sanity check orientamento (assiale / coronale / sagittale)

Un unico volume visto lungo i tre assi. Questa volta l'orientamento e'
verificato empiricamente rendendo le slice e ispezionando punti di
riferimento anatomici reali (corpo vertebrale = posteriore, asimmetria
cardiaca = sinistra del paziente, apice polmonare stretto vs base larga),
non solo derivato dai codici di orientamento -- un tentativo puramente
simbolico precedente si e' rivelato sbagliato.

Risultato verificato per il tensore `(D, H, W)` prodotto dal fix in
`prepare_radjepa_volumes.py`:

- **assiale** (`ct[z]`, forma `(H, W)`): gia' corretto cosi' com'e', nessuna
  trasformazione necessaria (anteriore in alto, destra del paziente a
  sinistra dello schermo).
- **coronale/sagittale** (`D` in verticale): serve `origin="lower"` perche'
  altrimenti l'asse D risulta capovolto (base polmonare larga in alto invece
  che in basso). Nessun mirror sull'asse orizzontale in questi due casi.
</cell id="cell-6">


In [ ]:
patient_id, timepoint, series_uid, path = sample[0]
volume = np.load(path).astype(np.float32, copy=False)
ct = volume[0]  # (D, H, W) = (S axis, A axis, R axis)

mid_d, mid_h, mid_w = (s // 2 for s in ct.shape)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
# Axial: ct[mid_d] is already (H, W) = (A, R), already correctly oriented.
axes[0].imshow(ct[mid_d], cmap="gray", vmin=0.0, vmax=1.0)
axes[0].set_title(f"axial (z={mid_d})")
# Coronal: fixing H (A axis) leaves (D, W) = (S, R) -- superior/inferior vs
# left/right. origin="lower" puts superior up (verified against lung
# apex/base taper); no horizontal mirror needed.
axes[1].imshow(ct[:, mid_h, :], cmap="gray", vmin=0.0, vmax=1.0, origin="lower", aspect="auto")
axes[1].set_title(f"coronal (y={mid_h})")
# Sagittal: fixing W (R axis) leaves (D, H) = (S, A) -- superior/inferior vs
# anterior/posterior. Same origin="lower" for superior up.
axes[2].imshow(ct[:, :, mid_w], cmap="gray", vmin=0.0, vmax=1.0, origin="lower", aspect="auto")
axes[2].set_title(f"sagittal (x={mid_w})")
for ax in axes:
    ax.axis("off")
fig.suptitle(f"{patient_id} / {timepoint} - orientation check")
plt.tight_layout()
plt.show()